<a href="https://colab.research.google.com/github/Bunkhuoch-Ann/Side_Quests_simple/blob/main/Underhill%26Waddell59_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Table 1**

In [1]:
import math
import numpy as np

def factorial_checked(k):
    """
    Return k! only for valid non-negative integer k.
    This catches impossible quantum-number combinations early.
    """
    if not isinstance(k, (int, np.integer)) or k < 0:
        raise ValueError(f"Factorial argument must be a non-negative integer, got {k}")

    return math.factorial(k)


from mpmath import mp

# More precision than ordinary float while we validate the calculation
mp.dps = 50


def psi_m(n_i, n_i_prime, m, n, n_prime):
    """
    Equation (12) of Underhill & Waddell (1959).

    Parameters
    ----------
    n_i, n_i_prime : int
        Either n1/n1' or n2/n2', depending on where psi_m appears
        in equations (9)-(11).

    m : int
        Non-negative |m| used in equation (12).

    n, n_prime : int
        Upper and lower principal quantum numbers, respectively.
        They must be different for a bound-bound transition.

    Returns
    -------
    mp.mpf
        psi_m(n_i, n_i_prime).
    """

    # ---- Basic validity checks ----
    integer_inputs = [n_i, n_i_prime, m, n, n_prime]

    if not all(isinstance(x, int) for x in integer_inputs):
        raise TypeError("All quantum numbers must be integers.")

    if n_i < 0 or n_i_prime < 0 or m < 0:
        raise ValueError("n_i, n_i_prime, and m must be non-negative.")

    if n <= 0 or n_prime <= 0:
        raise ValueError("n and n_prime must be positive.")

    if n == n_prime:
        raise ValueError("n and n_prime must differ for a spectral transition.")

    # This is the repeated factor in equation (12).
    Q = 4 * mp.mpf(n) * mp.mpf(n_prime) / (n - n_prime)**2

    # j = 0 term
    total = mp.mpf(1)
    term = mp.mpf(1)

    # The series terminates exactly: no infinite-series approximation.
    j_max = min(n_i, n_i_prime)

    for j in range(1, j_max + 1):

        # Generates the next term recursively:
        #
        # term_j / term_(j-1)
        #   = - [(n_i-j+1)(n_i'-j+1) / ((m+j) j)] Q
        #
        # This is algebraically equivalent to the factorial expression
        # in equation (12), but is more stable and simpler to code.
        term *= (
            - mp.mpf(n_i - j + 1)
            * mp.mpf(n_i_prime - j + 1)
            * Q
            / (mp.mpf(m + j) * mp.mpf(j))
        )

        total += term

    return total

from mpmath import mp

mp.dps = 50


def f_pi(n1_prime, n2_prime, m_prime,
         n1, n2, m,
         n, n_prime):
    """
    Equation (9): oscillator strength of one pi Stark component.

    Lower state: (n1_prime, n2_prime, m_prime)
    Upper state: (n1, n2, m)

    For a pi component, Delta m = 0, so m = m_prime.
    """

    # ------------------------------------------------------------
    # 1. Check that this is a pi component.
    # ------------------------------------------------------------
    if m != m_prime:
        raise ValueError(
            "Equation (9) is only for pi components: require m = m_prime."
        )

    # This paper's factorial expression uses non-negative |m|.
    M = abs(m)

    # ------------------------------------------------------------
    # 2. Check that the parabolic states are physically valid.
    #
    # n1 + n2 + |m| + 1 = n
    # n1' + n2' + |m'| + 1 = n'
    # ------------------------------------------------------------
    if n1 < 0 or n2 < 0 or n1_prime < 0 or n2_prime < 0:
        raise ValueError("All n1 and n2 quantum numbers must be non-negative.")

    if n1 + n2 + M + 1 != n:
        raise ValueError(
            "Upper state is invalid: n1 + n2 + |m| + 1 must equal n."
        )

    if n1_prime + n2_prime + M + 1 != n_prime:
        raise ValueError(
            "Lower state is invalid: n1' + n2' + |m'| + 1 must equal n'."
        )

    if n <= n_prime:
        raise ValueError(
            "For the absorption oscillator strength here, use upper n > lower n'."
        )

    # ------------------------------------------------------------
    # 3. Short names for the two repeated fractions.
    # ------------------------------------------------------------
    Q = (
        4 * mp.mpf(n) * mp.mpf(n_prime)
        / (n - n_prime)**2
    )

    R = (
        4 * mp.mpf(n) * mp.mpf(n_prime)
        / (n + n_prime)**2
    )

    # ------------------------------------------------------------
    # 4. Prefactor P in equation (9).
    # ------------------------------------------------------------
    prefactor = (
        mp.mpf(1) / 48
        * (mp.mpf(1) / n_prime**2 - mp.mpf(1) / n**2)
        * ((mp.mpf(n - n_prime) / (n + n_prime))
           ** (2 * (n + n_prime)))
        * Q**(2 * M + 4)
        / mp.factorial(M)**4
    )

    # ------------------------------------------------------------
    # 5. Factorial product D in equation (9).
    # ------------------------------------------------------------
    factorial_product = (
        mp.factorial(n1 + M) / mp.factorial(n1)
        * mp.factorial(n2 + M) / mp.factorial(n2)
        * mp.factorial(n1_prime + M) / mp.factorial(n1_prime)
        * mp.factorial(n2_prime + M) / mp.factorial(n2_prime)
    )

    # ------------------------------------------------------------
    # 6. Evaluate the ordinary psi factors.
    # ------------------------------------------------------------
    psi_11 = psi_m(n1, n1_prime, M, n, n_prime)
    psi_22 = psi_m(n2, n2_prime, M, n, n_prime)

    # ------------------------------------------------------------
    # 7. Evaluate B, the expression inside curly braces.
    # ------------------------------------------------------------
    first_part = (
        2 * (n1_prime - n2_prime)
        * (n**2 + n_prime**2) / (n + n_prime)**2
        - (n1 - n2) * R
    ) * psi_11 * psi_22

    # Do not call psi_m(..., -1, ...) when n1' or n2' is zero.
    # Mathematically those terms are zero because they are multiplied
    # by n1' or n2', respectively.
    shifted_n1_term = (
        mp.mpf(0)
        if n1_prime == 0
        else n1_prime
        * psi_m(n1, n1_prime - 1, M, n, n_prime)
        * psi_22
    )

    shifted_n2_term = (
        mp.mpf(0)
        if n2_prime == 0
        else n2_prime
        * psi_11
        * psi_m(n2, n2_prime - 1, M, n, n_prime)
    )

    bracket = first_part - 2 * (shifted_n1_term - shifted_n2_term)

    # ------------------------------------------------------------
    # 8. Final oscillator strength f_pi.
    # ------------------------------------------------------------
    return prefactor * factorial_product * bracket**2

def f_sigma_plus(n1_prime, n2_prime, m_prime,
                 n1, n2, m,
                 n, n_prime):
    """
    Equation (10): oscillator strength of one sigma-plus Stark component.

    Lower state: (n1_prime, n2_prime, m_prime)
    Upper state: (n1, n2, m)

    This component requires:
        m_prime - m = +1

    Important:
    The psi bracket below follows the user's current transcription:
        psi_m(n2, n1_prime) * psi_m(n2, n2_prime)
        - C^2 * psi_m(n2, n2_prime + 1)

    In particular, the second term contains only ONE psi factor.
    """

    # ------------------------------------------------------------
    # 1. Check the sigma-plus selection rule.
    # ------------------------------------------------------------
    if m_prime - m != 1:
        raise ValueError(
            "Equation (10) is for sigma-plus components: require m_prime - m = +1."
        )

    # Equation (10) contains m! and m'!, so use non-negative m values.
    if m < 0 or m_prime < 0:
        raise ValueError(
            "This implementation expects non-negative m and m_prime."
        )

    M = m
    M_prime = m_prime

    # ------------------------------------------------------------
    # 2. Check the parabolic quantum-number relations.
    # ------------------------------------------------------------
    if n1 < 0 or n2 < 0 or n1_prime < 0 or n2_prime < 0:
        raise ValueError("All n1 and n2 quantum numbers must be non-negative.")

    if n1 + n2 + M + 1 != n:
        raise ValueError(
            "Upper state is invalid: n1 + n2 + m + 1 must equal n."
        )

    if n1_prime + n2_prime + M_prime + 1 != n_prime:
        raise ValueError(
            "Lower state is invalid: n1_prime + n2_prime + m_prime + 1 "
            "must equal n_prime."
        )

    if n <= n_prime:
        raise ValueError(
            "Use upper n > lower n_prime for this absorption oscillator strength."
        )

    # ------------------------------------------------------------
    # 3. Repeated factors.
    # ------------------------------------------------------------
    Q = (
        4 * mp.mpf(n) * mp.mpf(n_prime)
        / (n - n_prime)**2
    )

    C = (
        mp.mpf(n - n_prime)
        / (n + n_prime)
    )

    # ------------------------------------------------------------
    # 4. Prefactor in equation (10).
    # ------------------------------------------------------------
    prefactor = (
        mp.mpf(1) / 48
        * (mp.mpf(1) / n_prime**2 - mp.mpf(1) / n**2)
        * C**(2 * (n + n_prime))
        / mp.factorial(M)**4
        * Q**(2 * M + 4)
    )

    # ------------------------------------------------------------
    # 5. Factorial product.
    #
    # The last denominator is assumed to be n2_prime!, following
    # the symmetric factorial pattern. Compare this against the PDF.
    # ------------------------------------------------------------
    factorial_product = (
        mp.factorial(n1 + M) / mp.factorial(n1)
        * mp.factorial(n2 + M) / mp.factorial(n2)
        * mp.factorial(n1_prime + M_prime) / mp.factorial(n1_prime)
        * mp.factorial(n2_prime + M_prime) / mp.factorial(n2_prime)
        # * mp.factorial(n2_prime + M_prime) / n2_prime
    )

    # ------------------------------------------------------------
    # 6. Curly-bracket expression.
    #
    # This is deliberately written from your current transcription.
    # Do not add another psi factor to the second term.
    # ------------------------------------------------------------
    first_term = (
        # psi_m(n2, n1_prime, M, n, n_prime)
        psi_m(n1, n1_prime, M, n, n_prime)  #recommended by gpt due to symmetry
        * psi_m(n2, n2_prime, M, n, n_prime)
    )

    second_term = (
        C**2
        * psi_m(n1, n1_prime + 1, M, n, n_prime) #recommended by gpt due to symmetry
        * psi_m(n2, n2_prime + 1, M, n, n_prime)
    )

    bracket = first_term - second_term

    # ------------------------------------------------------------
    # 7. Final sigma-plus oscillator strength.
    # ------------------------------------------------------------
    return prefactor * factorial_product * bracket**2

def f_sigma_minus(n1_prime, n2_prime, m_prime,
                  n1, n2, m,
                  n, n_prime):
    """
    Equation (11): oscillator strength of one sigma-minus Stark component.

    Lower state: (n1_prime, n2_prime, m_prime)
    Upper state: (n1, n2, m)

    This component requires:
        m_prime - m = -1
    """

    # ------------------------------------------------------------
    # 1. Check the sigma-minus selection rule.
    # ------------------------------------------------------------
    if m_prime - m != -1:
        raise ValueError(
            "Equation (11) is for sigma-minus components: "
            "require m_prime - m = -1."
        )

    # Equation (11) contains m! and m'!, so use non-negative values.
    if m < 0 or m_prime < 0:
        raise ValueError(
            "This implementation expects non-negative m and m_prime."
        )

    M = m
    M_prime = m_prime

    # ------------------------------------------------------------
    # 2. Check valid parabolic quantum-number states.
    # ------------------------------------------------------------
    if n1 < 0 or n2 < 0 or n1_prime < 0 or n2_prime < 0:
        raise ValueError("All n1 and n2 quantum numbers must be non-negative.")

    if n1 + n2 + M + 1 != n:
        raise ValueError(
            "Upper state is invalid: n1 + n2 + m + 1 must equal n."
        )

    if n1_prime + n2_prime + M_prime + 1 != n_prime:
        raise ValueError(
            "Lower state is invalid: n1_prime + n2_prime + m_prime + 1 "
            "must equal n_prime."
        )

    if n <= n_prime:
        raise ValueError(
            "Use upper n > lower n_prime for this absorption oscillator strength."
        )

    # ------------------------------------------------------------
    # 3. Repeated factors.
    # ------------------------------------------------------------
    Q = (
        4 * mp.mpf(n) * mp.mpf(n_prime)
        / (n - n_prime)**2
    )

    C = (
        mp.mpf(n - n_prime)
        / (n + n_prime)
    )

    # ------------------------------------------------------------
    # 4. Prefactor in equation (11).
    # Note: it uses m_prime, not m.
    # ------------------------------------------------------------
    prefactor = (
        mp.mpf(1) / 48
        * (mp.mpf(1) / n_prime**2 - mp.mpf(1) / n**2)
        * C**(2 * (n + n_prime))
        / mp.factorial(M_prime)**4
        * Q**(2 * M_prime + 4)
    )

    # ------------------------------------------------------------
    # 5. Factorial product.
    # ------------------------------------------------------------
    factorial_product = (
        mp.factorial(n1 + M) / mp.factorial(n1)
        * mp.factorial(n2 + M) / mp.factorial(n2)
        * mp.factorial(n1_prime + M_prime) / mp.factorial(n1_prime)
        * mp.factorial(n2_prime + M_prime) / mp.factorial(n2_prime)
    )

    # ------------------------------------------------------------
    # 6. Curly-bracket expression in equation (11).
    # ------------------------------------------------------------
    first_term = (
        psi_m(n1, n1_prime, M_prime, n, n_prime)
        * psi_m(n2, n2_prime, M_prime, n, n_prime)
    )

    second_term = (
        C**2
        * psi_m(n1 + 1, n1_prime, M_prime, n, n_prime)
        * psi_m(n2 + 1, n2_prime, M_prime, n, n_prime)
    )

    bracket = first_term - second_term

    # ------------------------------------------------------------
    # 7. Final sigma-minus oscillator strength.
    # ------------------------------------------------------------
    return prefactor * factorial_product * bracket**2

def X_k(n1_prime, n2_prime,
        n1, n2,
        n, n_prime):
    """
    Equation (8): dimensionless Stark displacement factor X_k.

    Lower state: (n1_prime, n2_prime)
    Upper state: (n1, n2)

    Returns
    -------
    int
        X_k = n(n2 - n1) - n_prime(n2_prime - n1_prime)
    """

    values = [n1_prime, n2_prime, n1, n2, n, n_prime]

    if not all(isinstance(value, int) for value in values):
        raise TypeError("All inputs must be integers.")

    if any(value < 0 for value in [n1_prime, n2_prime, n1, n2]):
        raise ValueError("n1, n2, n1_prime, and n2_prime must be non-negative.")

    if n <= 0 or n_prime <= 0:
        raise ValueError("n and n_prime must be positive.")

    return (
        n * (n2 - n1)
        - n_prime * (n2_prime - n1_prime)
    )

**Testing the correctness of the equations**

In [2]:
f_pi_1 = f_pi(
    n1_prime=0, n2_prime=0, m_prime=0,
    n1=3, n2=10, m=0,
    n=14, n_prime=1
)

print("First pi component :", mp.nstr(f_pi_1, 15))

First pi component : 1.03636440326646e-5


In [3]:
f_sigma_plus_1 = f_sigma_plus(
    n1_prime=0, n2_prime=0, m_prime=1,
    n1=1, n2=1, m=0,
    n=3, n_prime=2)
print("First sigma-plus component :", mp.nstr(f_sigma_plus_1, 15))

First sigma-plus component : 0.0399532621824


In [4]:
f_sigma_minus_1 = f_sigma_minus(
    n1_prime=0, n2_prime=0, m_prime=0,
    n1=0, n2=3, m=1,
    n=5, n_prime=1)
print("First sigma-minus component :", mp.nstr(f_sigma_minus_1, 15))

First sigma-minus component : 0.000464611462508375


In [5]:
X = X_k(
    n1_prime=0, n2_prime=0,
    n1=0, n2=2,
    n=3, n_prime=1
)

print(X)   # -2

6


**Building Table1**

In [6]:
import pandas as pd
from mpmath import mp

# ============================================================
# User controls
# ============================================================
n_max = 30

# False: keep both X < 0 and X >= 0 components.
# True : keep only X >= 0, matching the positive-X table convention.
ignore_negative_x = True

# The four series tabulated in the paper.
lower_levels = (1, 2, 3, 4)

# Components smaller than this are treated as exactly zero.
zero_tolerance = mp.mpf("1e-40")


# ============================================================
# Valid reduced parabolic states used by the current f functions.
#
# n1 + n2 + m + 1 = n, with m >= 0.
# ============================================================
def parabolic_states_reduced(n):
    states = []

    for m in range(n):
        for n1 in range(n - m):
            n2 = n - m - 1 - n1
            states.append((n1, n2, m))

    return states


# ============================================================
# Select the appropriate equation from m' - m.
# ============================================================
def component_f(n1_prime, n2_prime, m_prime,
                n1, n2, m,
                n, n_prime):

    delta_m_paper = m_prime - m

    if delta_m_paper == 0:
        return f_pi(
            n1_prime, n2_prime, m_prime,
            n1, n2, m,
            n, n_prime
        )

    elif delta_m_paper == +1:
        return f_sigma_plus(
            n1_prime, n2_prime, m_prime,
            n1, n2, m,
            n, n_prime
        )

    elif delta_m_paper == -1:
        return f_sigma_minus(
            n1_prime, n2_prime, m_prime,
            n1, n2, m,
            n, n_prime
        )

    else:
        raise ValueError(
            "This pair is not dipole-allowed: require m_prime - m = 0, +1, or -1."
        )


# ============================================================
# Build Table I
# ============================================================
def build_table_I(n_max=30,
                  lower_levels=(1, 2, 3, 4),
                  ignore_negative_x=True,
                  zero_tolerance=mp.mpf("1e-40")):

    rows = []

    for n_prime in lower_levels:

        # A bound-bound absorption transition requires n > n'.
        for n in range(n_prime + 1, n_max + 1):

            lower_states = parabolic_states_reduced(n_prime)
            upper_states = parabolic_states_reduced(n)

            for n1_prime, n2_prime, m_prime in lower_states:
                for n1, n2, m in upper_states:

                    # Electric-dipole selection rule:
                    # m' - m = 0, +1, or -1
                    if (m_prime - m) not in (0, +1, -1):
                        continue

                    X = X_k(
                        n1_prime, n2_prime,
                        n1, n2,
                        n, n_prime
                    )

                    # Optional positive-X-only convention.
                    if ignore_negative_x and X < 0:
                        continue

                    f_value = component_f(
                        n1_prime, n2_prime, m_prime,
                        n1, n2, m,
                        n, n_prime
                    )

                    # Do not include formally allowed but zero-strength components.
                    if abs(f_value) <= zero_tolerance:
                        continue

                    rows.append({
                        "n'": n_prime,
                        "n": n,
                        "n_1'": n1_prime,
                        "n_2'": n2_prime,
                        "m'": m_prime,
                        "n_1": n1,
                        "n_2": n2,
                        "m": m,
                        # Store as text so the CSV keeps high precision.
                        "f": mp.nstr(f_value, 30),
                        "X": X,
                    })

    table_I = pd.DataFrame(rows)

    # Order rows for easy comparison with the printed table.
    table_I = table_I.sort_values(
        by=["n'", "n", "X", "m'", "m", "n_1'", "n_2'", "n_1", "n_2"]
    ).reset_index(drop=True)

    return table_I


# ============================================================
# Create and save the CSV
# ============================================================
table_I = build_table_I(
    n_max=n_max,
    lower_levels=lower_levels,
    ignore_negative_x=ignore_negative_x,
    zero_tolerance=zero_tolerance
)

suffix = "positive_X_only" if ignore_negative_x else "all_X"
output_csv = f"/content/table_I_nmax_{n_max}_{suffix}.csv"

table_I.to_csv(output_csv, index=False)

print(f"Saved {len(table_I):,} components to:")
print(output_csv)

display(table_I.head(20))

Saved 10,740 components to:
/content/table_I_nmax_30_positive_X_only.csv


,n',n,n_1',n_2',m',n_1,n_2,m,f,X
0,1,2,0,0,0,0,0,1,0.0693661196633304543684059001846,0
1,1,2,0,0,0,0,1,0,0.0693661196633304543684059001846,2
2,1,3,0,0,0,0,1,1,0.006591796875,3
3,1,3,0,0,0,0,2,0,0.01318359375,6
4,1,4,0,0,0,1,1,1,0.0019327352832,0
5,1,4,0,0,0,1,2,0,0.0004831838208,4
6,1,4,0,0,0,0,2,1,0.0014495514624,8
7,1,4,0,0,0,0,3,0,0.0043486543872,12
8,1,5,0,0,0,1,2,1,0.000696917193762563239137308507192,5
9,1,5,0,0,0,1,3,0,0.000464611462508375492758205671461,10


In [7]:
import pandas as pd
from mpmath import mp

# ------------------------------------------------------------
# User setting
#
# True  -> table_I contains only X >= 0 representatives.
# False -> table_I contains both positive and negative X rows.
# ------------------------------------------------------------
table_I_has_positive_X_only = True


def component_weight(m_prime, m):
    """
    Weight rule stated below equation (13).

    sigma component: m' - m = ±1  -> weight 4
    pi component:    m' = m != 0 -> weight 2
    pi component:    m' = m = 0  -> weight 1
    """

    delta = m_prime - m

    if delta in (+1, -1):
        return mp.mpf(4)

    if delta == 0:
        return mp.mpf(1) if m_prime == 0 else mp.mpf(2)

    raise ValueError(
        f"Invalid component: m_prime - m = {delta}, "
        "but it must be 0, +1, or -1."
    )


def build_table_II(table_I, table_I_has_positive_X_only=True):
    """
    Reconstruct Table II from Table I using equation (13).

    Required Table-I columns:
    n', n, m', m, f, X

    Returns columns:
    n', n, f_0, f_pm, f
    """

    required_columns = {"n'", "n", "m'", "m", "f", "X"}
    missing = required_columns - set(table_I.columns)

    if missing:
        raise ValueError(f"table_I is missing columns: {sorted(missing)}")

    rows = []

    # One Table-II row for each hydrogen line n' -> n.
    for (n_prime, n), group in table_I.groupby(["n'", "n"], sort=True):

        n_prime = int(n_prime)

        weighted_positive_X = mp.mpf(0)
        weighted_negative_X = mp.mpf(0)
        weighted_zero_X = mp.mpf(0)

        for _, row in group.iterrows():

            m_prime = int(row["m'"])
            m = int(row["m"])
            X = int(row["X"])

            # f was saved as a high-precision decimal string in Table I.
            f_component = mp.mpf(str(row["f"]))

            weight = component_weight(m_prime, m)
            weighted_f = weight * f_component

            if X > 0:
                weighted_positive_X += weighted_f

            elif X < 0:
                weighted_negative_X += weighted_f

            else:
                weighted_zero_X += weighted_f

        # --------------------------------------------------------
        # Shifted contribution: f_pm
        # --------------------------------------------------------
        if table_I_has_positive_X_only:

            # The negative-X side is omitted but has equal strength.
            f_pm = 2 * weighted_positive_X / n_prime**2

        else:
            # Both X > 0 and X < 0 rows are already present.
            f_pm = (
                weighted_positive_X + weighted_negative_X
            ) / n_prime**2

        # --------------------------------------------------------
        # Unshifted contribution: f_0
        # --------------------------------------------------------
        f_0 = weighted_zero_X / n_prime**2

        # Equation (2) and equation (13)
        f_total = f_pm + f_0

        rows.append({
            "n'": n_prime,
            "n": int(n),

            # Save many digits for later checks.
            "f_0": mp.nstr(f_0, 30),
            "f_pm": mp.nstr(f_pm, 30),
            "f": mp.nstr(f_total, 30),
        })

    table_II = pd.DataFrame(rows)

    return table_II.sort_values(
        by=["n'", "n"]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# Build and save Table II
# ------------------------------------------------------------
table_II = build_table_II(
    table_I,
    table_I_has_positive_X_only=table_I_has_positive_X_only
)

suffix = (
    "from_positive_X_table_I"
    if table_I_has_positive_X_only
    else "from_all_X_table_I"
)

output_csv = f"/content/table_II_nmax_30_{suffix}.csv"

table_II.to_csv(output_csv, index=False)

print(f"Saved Table II to:\n{output_csv}")
display(table_II.head(20))

Saved Table II to:
/content/table_II_nmax_30_from_positive_X_table_I.csv


,n',n,f_0,f_pm,f
0,1,2,0.277464478653321817473623600738,0.138732239326660908736811800369,0.416196717979982726210435401108
1,1,3,0.0,0.0791015625,0.0791015625
2,1,4,0.0077309411328,0.0212600881152,0.028991029248
3,1,5,0.0,0.0139383438752512647827461701438,0.0139383438752512647827461701438
4,1,6,0.00133705589464116885397134304208,0.00646243682409898279419482470337,0.00779949271874015164816616774545
5,1,7,0.0,0.00481395081442315131425857543945,0.00481395081442315131425857543945
6,1,8,0.000404244737081423209475399575552,0.00277918256743478456514337208191,0.00318342730451620777461877165747
7,1,9,0.0,0.0022161087866383368192,0.0022161087866383368192
8,1,10,0.000162158315114508042881766372012,0.0014432090045191215816477207109,0.00160536731963362962452948708291
9,1,11,0.0,0.00120052093871665957398507394832,0.00120052093871665957398507394832


In [8]:
import pandas as pd
from mpmath import mp
from astropy.constants import h, c as c_light, m_e, e

mp.dps = 50

# ------------------------------------------------------------
# Modern physical constants in Gaussian-cgs units.
# Equation (7) is written in this unit system.
# ------------------------------------------------------------
H_CGS = mp.mpf(str(h.cgs.value))          # erg s
C_CGS = mp.mpf(str(c_light.cgs.value))    # cm s^-1
ME_CGS = mp.mpf(str(m_e.cgs.value))       # g
E_ESU = mp.mpf(str(e.esu.value))          # statcoulomb

CM_TO_ANGSTROM = mp.mpf("1e8")


def component_weight(m_prime, m):
    """
    Weight convention stated below equation (13).

    sigma: m' - m = ±1  -> 4
    pi:    m' = m != 0 -> 2
    pi:    m' = m = 0  -> 1
    """

    delta = m_prime - m

    if delta in (+1, -1):
        return mp.mpf(4)

    if delta == 0:
        return mp.mpf(1) if m_prime == 0 else mp.mpf(2)

    raise ValueError(
        f"Invalid transition: m_prime - m = {delta}."
    )


def c_k_modern(n_prime, n, X, Z=1):
    """
    Equation (7), using modern constants.

    Returns c_k in Angstrom per electrostatic field unit, so that:

        Delta_lambda_k [Angstrom] = c_k * F [esu]

    For Table III, use Z=1 for hydrogen.
    """

    n_prime = mp.mpf(n_prime)
    n = mp.mpf(n)
    X = mp.mpf(X)
    Z = mp.mpf(Z)

    prefactor = (
        3 * H_CGS**7 * C_CGS
        / (32 * mp.pi**6 * ME_CGS**3 * E_ESU**9)
    )

    line_factor = (
        n_prime**4 * n**4
        / (n**2 - n_prime**2)**2
    )

    c_k_cgs = prefactor * line_factor * X / Z**5

    # Equation (7) naturally gives a length in cm.
    # Convert it to Angstrom for convenient modern output.
    return c_k_cgs * CM_TO_ANGSTROM


def build_table_III(table_I, Z=1):
    """
    Construct Table III from Table I.

    Table III contains positive-X rows only.

    For every (n', n, X > 0):
        partial_f_X = [sum(weight_k * f_k)] / n'^2

    No extra factor of 2 is used here.
    The missing negative-X side is restored only later when making f_pm:
        f_pm = 2 * sum(partial_f_X for X > 0)
    """

    required_columns = {"n'", "n", "m'", "m", "f", "X"}
    missing = required_columns - set(table_I.columns)

    if missing:
        raise ValueError(f"table_I is missing columns: {sorted(missing)}")

    # Table III lists only positive X.
    table_positive_X = table_I[
        table_I["X"].astype(int) > 0
    ].copy()

    rows = []

    # Group every component sharing the same line and same X.
    for (n_prime, n, X), group in table_positive_X.groupby(
        ["n'", "n", "X"],
        sort=True
    ):

        n_prime = int(n_prime)
        n = int(n)
        X = int(X)

        weighted_sum = mp.mpf(0)

        for _, row in group.iterrows():

            m_prime = int(row["m'"])
            m = int(row["m"])

            # Works whether f was stored as text or as a float.
            f_component = mp.mpf(str(row["f"]))

            weight = component_weight(m_prime, m)

            weighted_sum += weight * f_component

        # This is Table III column 5.
        # partial_f = weighted_sum / n_prime**2
        partial_f = weighted_sum

        # This is Table III column 3, in modern notation.
        c_k = c_k_modern(
            n_prime=n_prime,
            n=n,
            X=X,
            Z=Z
        )

        rows.append({
            "n'": n_prime,
            "n": n,
            "c_k_A_per_esu": mp.nstr(c_k, 20),
            "X": X,
            "f_partial": mp.nstr(partial_f, 30),
        })

    table_III = pd.DataFrame(rows)

    return table_III.sort_values(
        by=["n'", "n", "X"]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# Build and save Table III for hydrogen.
# ------------------------------------------------------------
table_III = build_table_III(
    table_I=table_I,
    Z=1
)

output_csv = "/content/table_III_nmax_30_modern_constants.csv"

table_III.to_csv(output_csv, index=False)

print(f"Saved {len(table_III):,} Table-III rows to:")
print(output_csv)

display(table_III.head(20))

Saved 6,788 Table-III rows to:
/content/table_III_nmax_30_modern_constants.csv


,n',n,c_k_A_per_esu,X,f_partial
0,1,2,0.00056669001595757108008,2,0.0693661196633304543684059001846
1,1,3,0.00060515188715781638288,3,0.0263671875
2,1,3,0.0012103037743156327658,6,0.01318359375
3,1,4,0.0007253632204256909825,4,0.0004831838208
4,1,4,0.001450726440851381965,8,0.0057982058496
5,1,4,0.0021760896612770729475,12,0.0043486543872
6,1,5,0.00086470034173213360608,5,0.00278766877505025295654923402877
7,1,5,0.0017294006834642672122,10,0.000464611462508375492758205671461
8,1,5,0.0025941010251964008182,15,0.00185844585003350197103282268584
9,1,5,0.0034588013669285344243,20,0.00185844585003350197103282268584
